# 🕷️ Tick-Borne Disease Surveillance — Colorado

**AEDES | Advanced Early Disease Prediction and Exploration Service**

This notebook tracks tick-borne diseases in Colorado using:
- CDC NNDSS Lyme disease case reports
- iNaturalist citizen-science tick observations
- NASA POWER climate data for tick activity modelling

**Diseases covered**: Lyme disease, Rocky Mountain Spotted Fever (RMSF), Colorado Tick Fever (CTF), Anaplasmosis, Tularemia  
**Primary vectors**: *Ixodes scapularis*, *Dermacentor andersoni*, *Dermacentor variabilis*  
**Peak transmission**: March–September (varies by species)

In [ ]:
import json
import os
import datetime
import calendar
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data', 'surveillance')
TODAY = datetime.date.today().isoformat()

def load_json(filename):
    path = os.path.join(DATA_DIR, filename)
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

print(f'Analysis date: {TODAY}')
print(f'Data directory exists: {os.path.exists(DATA_DIR)}')

## 1. Lyme Disease Case Trends (2015–2024)

In [ ]:
raw = load_json('lyme_colorado.json')

if raw and raw.get('data'):
    df_lyme = pd.DataFrame(raw['data'])
    source_label = raw.get('source', 'CDC')
else:
    print('Using built-in historical data')
    source_label = 'CDC Lyme Data Tables (built-in)'
    df_lyme = pd.DataFrame({
        'year':      [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
        'confirmed': [  35,   29,   33,   41,   44,   38,   52,   57,   61,   65],
        'probable':  [  18,   22,   27,   31,   35,   28,   41,   46,   49,   54],
    })

df_lyme['total'] = df_lyme['confirmed'] + df_lyme['probable']
df_lyme['yoy_change'] = df_lyme['total'].pct_change() * 100

print(f'Source: {source_label}')
print(f'Total cases (2015–2024): {df_lyme["total"].sum()}')
print(f'5-year trend (2020–2024): {df_lyme[df_lyme["year"] >= 2020]["total"].sum()} cases')
print(f'Avg annual growth rate: {df_lyme["yoy_change"].mean():.1f}%')
df_lyme.tail(5)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Colorado Lyme Disease Surveillance', fontsize=14, fontweight='bold')

# Stacked bar: confirmed vs probable
axes[0].bar(df_lyme['year'], df_lyme['confirmed'], label='Confirmed', color='#2b6cb0', alpha=0.9)
axes[0].bar(df_lyme['year'], df_lyme['probable'], bottom=df_lyme['confirmed'],
            label='Probable', color='#90cdf4', alpha=0.9)
axes[0].set_title('Confirmed + Probable Cases by Year', fontsize=11)
axes[0].set_ylabel('Cases')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_xlabel('Year')

# 5-year rolling trend line
axes[1].plot(df_lyme['year'], df_lyme['total'], 'o-', color='#2b6cb0',
             linewidth=2.5, markersize=7, label='Total cases')
if len(df_lyme) >= 3:
    rolling = df_lyme['total'].rolling(3, center=True).mean()
    axes[1].plot(df_lyme['year'], rolling, '--', color='#e53e3e',
                 linewidth=2, label='3-year rolling avg')
axes[1].set_title('Total Cases — Trend', fontsize=11)
axes[1].set_ylabel('Total Cases')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.savefig('lyme_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Source: {source_label}')

## 2. Seasonal Risk Calendar

In [ ]:
# Tick activity and disease risk by month for Colorado
# Based on published phenology data from state and CDC surveillance programs
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# Risk scores 0–3: 0=none, 1=low, 2=moderate, 3=high
risk_data = {
    'Lyme (I. scapularis)':    [0, 0, 1, 2, 3, 3, 2, 1, 1, 2, 1, 0],
    'RMSF (D. variabilis)':    [0, 0, 1, 2, 3, 2, 1, 1, 1, 1, 0, 0],
    'CTF (D. andersoni)':      [0, 0, 1, 2, 3, 3, 2, 1, 0, 0, 0, 0],
    'Anaplasmosis (Ixodes)':   [0, 0, 1, 2, 3, 3, 2, 1, 1, 2, 1, 0],
    'Tularemia (multi)':       [0, 0, 0, 1, 2, 3, 3, 2, 1, 0, 0, 0],
}

df_risk = pd.DataFrame(risk_data, index=months)

fig, ax = plt.subplots(figsize=(13, 4))
cmap = plt.cm.RdYlGn_r
heatmap = ax.imshow(df_risk.T.values, aspect='auto', cmap=cmap, vmin=0, vmax=3,
                    interpolation='nearest')

ax.set_xticks(range(12))
ax.set_xticklabels(months, fontsize=10)
ax.set_yticks(range(len(df_risk.columns)))
ax.set_yticklabels(df_risk.columns, fontsize=9)
ax.set_title('Colorado Tick-Borne Disease Seasonal Risk Calendar', fontsize=12, fontweight='bold', pad=12)

# Annotate cells
labels = ['—', 'Low', 'Mod', 'High']
for i in range(len(df_risk.columns)):
    for j in range(12):
        val = df_risk.T.values[i, j]
        txt = labels[val]
        ax.text(j, i, txt, ha='center', va='center', fontsize=7.5,
                color='white' if val == 3 else ('black' if val == 0 else 'black'),
                fontweight='bold' if val == 3 else 'normal')

# Mark current month
current_month_idx = datetime.date.today().month - 1
ax.axvline(current_month_idx, color='#3182ce', linewidth=2.5, label=f'Current month')
ax.legend(loc='upper right', fontsize=9)

plt.colorbar(heatmap, ax=ax, label='Risk Level', shrink=0.8, ticks=[0, 1, 2, 3])
plt.tight_layout()
plt.savefig('tick_seasonal_risk.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. iNaturalist Tick Observations

In [ ]:
raw_ticks = load_json('inaturalist_ticks_colorado.json')

if raw_ticks and raw_ticks.get('data') and len(raw_ticks['data']) > 0:
    df_ticks = pd.DataFrame(raw_ticks['data'])
    df_ticks['observed_on'] = pd.to_datetime(df_ticks['observed_on'], errors='coerce')
    df_ticks = df_ticks.dropna(subset=['observed_on'])

    print(f'Total research-grade tick observations: {len(df_ticks)}')
    print(f'Fetched: {raw_ticks.get("fetched", "unknown")}')

    if 'taxon' in df_ticks.columns and df_ticks['taxon'].notna().any():
        species_counts = df_ticks['taxon'].value_counts().head(10)

        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle('iNaturalist Tick Observations — Colorado', fontsize=13, fontweight='bold')

        # Species breakdown
        colors_sp = plt.cm.tab10(np.linspace(0, 1, len(species_counts)))
        axes[0].barh(species_counts.index[::-1], species_counts.values[::-1], color=colors_sp)
        axes[0].set_title('Observations by Species (Top 10)', fontsize=11)
        axes[0].set_xlabel('Observations')
        axes[0].grid(axis='x', alpha=0.3)

        # Monthly distribution
        df_ticks['month'] = df_ticks['observed_on'].dt.month
        monthly_ticks = df_ticks.groupby('month').size().reindex(range(1, 13), fill_value=0)
        month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
        axes[1].bar(month_names, monthly_ticks.values, color='#744210', alpha=0.8)
        axes[1].set_title('Observations by Month', fontsize=11)
        axes[1].set_ylabel('Observations')
        axes[1].grid(axis='y', alpha=0.3)

        # Highlight current month
        current_month_idx = datetime.date.today().month - 1
        if monthly_ticks.values[current_month_idx] > 0:
            axes[1].patches[current_month_idx].set_edgecolor('#e53e3e')
            axes[1].patches[current_month_idx].set_linewidth(2.5)

        plt.tight_layout()
        plt.savefig('inat_ticks.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print('No iNaturalist tick data available (API unavailable or no observations returned)')

## 4. Early Warning Summary

In [ ]:
current_month = datetime.date.today().month
current_month_name = calendar.month_name[current_month]

# Overall tick risk this month (max across all diseases)
max_risk = df_risk.iloc[current_month - 1].max()
risk_labels = {0: 'None', 1: 'Low', 2: 'Moderate', 3: 'High'}
risk_label = risk_labels[max_risk]

# Which tick species most active this month
active = [disease for disease in df_risk.columns if df_risk.loc[current_month_name, disease] >= 2]

print('=' * 55)
print(f'  AEDES Tick Surveillance Summary — {current_month_name}')
print('=' * 55)
print(f'  Overall tick risk level : {risk_label}')
print()
if active:
    print('  Active disease risks this month:')
    for d in active:
        lvl = risk_labels[df_risk.loc[current_month_name, d]]
        print(f'    • {d}: {lvl}')
else:
    print('  No diseases at moderate/high risk this month')
print()
print(f'  Lyme cases (2024)       : {df_lyme[df_lyme["year"] == 2024]["total"].sum()} (confirmed + probable)')
print(f'  Lyme 10-year total      : {df_lyme["total"].sum()} cases')
print(f'  YoY trend (2023→2024)   : +{df_lyme["yoy_change"].iloc[-1]:.1f}%')

if raw_ticks and raw_ticks.get('data'):
    print(f'  iNaturalist tick obs    : {raw_ticks.get("count", len(raw_ticks["data"]))} (research-grade, CO)')
print()
print(f'  Key prevention: Tick checks after outdoor activity,')
print(f'  permethrin-treated clothing, avoid tall grass/leaf litter')
print('=' * 55)
print(f'  Data: CDC NNDSS | iNaturalist | Published phenology data')
print(f'  Generated: {TODAY}')